# Chatbot integrado

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import sys
import os
from transformers import pipeline
sys.path.append(os.path.abspath('..'))
from src.busqueda_faiss import buscar_en_estrategia
from src.responder_ia import responder_ia
from src.Filtrador_finetuned import Filtrador_finetuned

In [2]:
modelo_e5 = SentenceTransformer('intfloat/multilingual-e5-small')

In [3]:
ruta_modelo = "../models/modelo_distilbeto_resenias"
polarizador = pipeline("text-classification", model=ruta_modelo, tokenizer=ruta_modelo)

Device set to use cpu


In [4]:
ruta_modelo = "../models/modelo_categorias_distilbeto"
clasificador = pipeline("text-classification", model=ruta_modelo, tokenizer=ruta_modelo)

Device set to use cpu


## Recuperacion de reseñas

In [6]:
lista_preguntas = [
    '¿A que lugar puedo ir para pasar buen tiempo en familia?', 
    '¿Angel es un buen guia?', 
    '¿Cuales son los peores tours?', 
    '¿Las atracciones estan limpias?', 
    '¿La neblina impide ver al volcan?', 
    '¿Los tours son muy largos?',
    'Quiero aprender de la historia de España ¿Que lugares me recomiendas?'
    ]

pregunta = lista_preguntas[1]

resultados_faiss = buscar_en_estrategia(
    pregunta= pregunta,
    nombre_estrategia='oraciones',
    modelo=modelo_e5,
    top_k=5,
)

contexto_recuperado = "\n\n".join([r["chunk"] for r in resultados_faiss])
print(contexto_recuperado)

El guía, Angel, es una maravilla, para la actividad tan densa que tiene que realizar es capaz de mantener el interés del grupo además de su propio entusiasmo por enseñar. Solo por poner una pega, estaría bien dedicar la mañana a un monumento y la tarde a otro dejando un descanso en medio para comer y descansar un rato.

Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos.

Angel es un guía simpático pero con muy poco conocimiento histórico. Habla muy poco de historia y mucho de cosas irrelevantes y repetitivas.

Angel es un guía a quien realmente le apasiona lo que hace. Le interesa que aprendamos y hacernos vivir una experiencia que nos inmersa en el tiempo, en los momentos.

Ángel es un guía fantástico.


## Filtrado

### Identificar pregunta

In [7]:
pregunta_polaridad = polarizador(pregunta) # 2= positivo, 1= neutro, 0 = negativo
pregunta_lugar = clasificador(pregunta)

print(pregunta_polaridad)
print(pregunta_lugar)

[{'label': 'LABEL_2', 'score': 0.9341174364089966}]
[{'label': 'Historicos', 'score': 0.763420820236206}]


In [8]:
filtro_polaridad = pregunta_polaridad[0]['label'] if pregunta_polaridad[0]['score'] >= 0.9 else False
filtro_lugar = pregunta_lugar[0]['label'] if pregunta_lugar[0]['score'] >= 0.9 else False

print(filtro_polaridad)
print(filtro_lugar)

LABEL_2
False


### Identificar reseñas

In [9]:
textos = [fila['chunk'] for fila in resultados_faiss]

categorias = clasificador(textos)
polaridades = polarizador(textos)

clasificaciones = pd.DataFrame([
    {
        'resena_id': fila['resena_id'],
        'cat_nombre': cat['label'],
        'cat_confi': cat['score'],
        'polar_nombre': pol['label'],
        'polar_confi': pol['score'],
        'chunk': fila['chunk']
    }
    for fila, cat, pol in zip(resultados_faiss, categorias, polaridades)
])

clasificaciones

,resena_id,cat_nombre,cat_confi,polar_nombre,polar_confi,chunk
0,33,Historicos,0.869664,LABEL_2,0.986709,"El guía, Angel, es una maravilla, para la acti..."
1,104,Historicos,0.883546,LABEL_2,0.988519,"Ángel es un guía fantástico. Atento, agradable..."
2,510,Historicos,0.937573,LABEL_1,0.889874,Angel es un guía simpático pero con muy poco c...
3,513,Historicos,0.932401,LABEL_2,0.987783,Angel es un guía a quien realmente le apasiona...
4,175,Aire libre,0.664754,LABEL_2,0.693349,Ángel es un guía fantástico.


In [10]:
dfs_a_unir = []

if filtro_lugar:
    score_suficiente = clasificaciones[clasificaciones['cat_confi'] > 0.80]
    lugar_filtrado = score_suficiente[score_suficiente['cat_nombre'] == filtro_lugar]
    dfs_a_unir.append(lugar_filtrado)

if filtro_polaridad:
    score_suficiente = clasificaciones[clasificaciones['polar_confi'] > 0.80]
    polar_filtrado = score_suficiente[score_suficiente['polar_nombre'] == filtro_polaridad]
    dfs_a_unir.append(polar_filtrado)

if dfs_a_unir:
    resenias_filtradas = pd.concat(dfs_a_unir, axis=0, ignore_index=True)
else:
    resenias_filtradas = clasificaciones

resenias_filtradas = resenias_filtradas.drop_duplicates()

with pd.option_context('display.max_colwidth', None):
    display(resenias_filtradas)

,resena_id,cat_nombre,cat_confi,polar_nombre,polar_confi,chunk
0,33,Historicos,0.869664,LABEL_2,0.986709,"El guía, Angel, es una maravilla, para la actividad tan densa que tiene que realizar es capaz de mantener el interés del grupo además de su propio entusiasmo por enseñar. Solo por poner una pega, estaría bien dedicar la mañana a un monumento y la tarde a otro dejando un descanso en medio para comer y descansar un rato."
1,104,Historicos,0.883546,LABEL_2,0.988519,"Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos."
2,513,Historicos,0.932401,LABEL_2,0.987783,"Angel es un guía a quien realmente le apasiona lo que hace. Le interesa que aprendamos y hacernos vivir una experiencia que nos inmersa en el tiempo, en los momentos."


In [11]:
og_df = pd.read_csv('../data/5k_metadata.csv', sep=';')
og_df = og_df[['business_name', 'review_rating', 'resena_id', 'categoria']]


df_metadata = pd.merge(
    og_df, 
    resenias_filtradas, 
    on='resena_id', 
    how='right'
)

df_metadata

,business_name,review_rating,resena_id,categoria,cat_nombre,cat_confi,polar_nombre,polar_confi,chunk
0,Visita guiada por el Museo del Prado y el Pala...,5,33,Historicos,Historicos,0.869664,LABEL_2,0.986709,"El guía, Angel, es una maravilla, para la acti..."
1,Visita guiada por el Museo del Prado y el Pala...,5,104,Historicos,Historicos,0.883546,LABEL_2,0.988519,"Ángel es un guía fantástico. Atento, agradable..."
2,Visita guiada por el Museo del Prado y el Pala...,5,513,Historicos,Historicos,0.932401,LABEL_2,0.987783,Angel es un guía a quien realmente le apasiona...


## Generador

In [12]:
contexto_csv = df_metadata[['business_name', 'chunk']].to_csv(
    index=False, 
    header=['lugar', 'reseña']
)
print(contexto_csv)

lugar,reseña
Visita guiada por el Museo del Prado y el Palacio Real,"El guía, Angel, es una maravilla, para la actividad tan densa que tiene que realizar es capaz de mantener el interés del grupo además de su propio entusiasmo por enseñar. Solo por poner una pega, estaría bien dedicar la mañana a un monumento y la tarde a otro dejando un descanso en medio para comer y descansar un rato."
Visita guiada por el Museo del Prado y el Palacio Real,"Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos."
Visita guiada por el Museo del Prado y el Palacio Real,"Angel es un guía a quien realmente le apasiona lo que hace. Le interesa que aprendamos y hacernos vivir una experiencia que nos inmersa en el tiempo, en los momentos."



In [20]:
modelos_locales = ['qwen2.5:7b', 'qwen2.5:3b', 'gemma2:2b', 'llama3.2:3b'] 

historial_chat = []

respuesta_1, historial_chat = responder_ia(
    pregunta=pregunta,
    contexto=contexto_csv,
    historial=historial_chat,
    modelo=modelos_locales[0]
)

print(f"PREGUNTA: {pregunta}\n")
print(f"RESPUESTA: \n{respuesta_1}")

PREGUNTA: ¿Angel es un buen guia?

RESPUESTA: 
Sí, Angel es un buen guía. Según las reseñas, Angel mantiene el interés del grupo, es atento y agradable, tiene muchos conocimientos y le apasiona lo que hace.


In [ ]:
pregunta2 = "¿Ese guia habla bien Inglés?"
contexto2 = pd.DataFrame({
    'lugar': [
        'Visita guiada por el Parque Guell', 
        'Visita guiada por el Museo del Prado y el Palacio Real'
    ],
    'reseña': [
        'Pude disfrutar del tour guiado, pero el guia Angel solo sabe Español y no se pudo comunicar con los extranjeros', 
        'La guia Melisa es muy habilidosa, se pudo comunicar con todos los turistas, incluso con los que hablaban ingles'
    ]
}).to_csv(index=False)

respuesta_2, historial_chat = responder_ia(
    pregunta=pregunta2,
    contexto=contexto2,
    historial=historial_chat,
    modelo=modelos_locales[0]
)

In [22]:
print(f"PREGUNTA: {pregunta2}\n")
print(f"RESPUESTA: \n{respuesta_2}")

PREGUNTA: ¿Ese guia habla bien Inglés?

RESPUESTA: 
No, Angel no habla inglés. Según la reseña, Angel solo sabe español y no se pudo comunicar con los extranjeros durante la visita guiada al Parque Guell.


## Pruebas

In [9]:
pregunta= lista_preguntas[3]

resultados_faiss = buscar_en_estrategia(
    pregunta= pregunta,
    nombre_estrategia='oraciones',
    modelo=modelo_e5,
    top_k=5,
)

filtrador = Filtrador_finetuned()
contexto_metadata, pregunta_metadata  = filtrador.procesar(pregunta= pregunta,resultados_faiss= resultados_faiss)

contexto_csv = contexto_metadata[['business_name', 'chunk']].to_csv(
    index=False, 
    header=['lugar', 'reseña']
)

historial_chat = []

respuesta, historial_chat = responder_ia(
    pregunta=pregunta,
    contexto=contexto_csv,
    historial=historial_chat,
    modelo=modelos_locales[3]
)

print(f"PREGUNTA: {pregunta}\n")
print(f"RESPUESTA: \n{respuesta}")

Device set to use cpu
Device set to use cpu


PREGUNTA: ¿Las atracciones estan limpias?

RESPUESTA: 
No, no todas las atracciones están limpias. En el Museo de los Niños Costa Rica, se menciona que algunas salas presentaban un intenso olor a moho y que el área de juegos de niños necesita mantenimiento urgente. En el caso de Kalambu Hot Springs, se menciona que el área de juegos de niños está sumamente sucia y desagradable.


In [40]:
pregunta = lista_preguntas[4]

resultados_faiss = buscar_en_estrategia(
    pregunta= pregunta,
    nombre_estrategia='oraciones',
    modelo=modelo_e5,
    top_k=5,
)

contexto_metadata, pregunta_metadata  = filtrador.procesar(pregunta= pregunta,resultados_faiss= resultados_faiss)

contexto_csv = contexto_metadata[['business_name', 'chunk']].to_csv(
    index=False, 
    header=['lugar', 'reseña']
)

historial_chat = []

respuesta, historial_chat = responder_ia(
    pregunta=pregunta,
    contexto=contexto_csv,
    historial=historial_chat,
    modelo=modelos_locales[0]
)

print(f"PREGUNTA: {pregunta}\n")
print(f"RESPUESTA: \n{respuesta}")

PREGUNTA: ¿La neblina impide ver al volcan?

RESPUESTA: 
Sí, según las reseñas proporcionadas, la neblina o nubes son un factor común que impide ver el volcán Arenal. Varios visitantes mencionan que el volcán no se ve debido a la nubosidad.
